In [1]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
index_df = pd.read_csv("features/feature_index_w3_s1.csv")

In [4]:
K = 60  # vocabulary size

X_hist = []
y_rec = []

for _, row in index_df.iterrows():
    subject_id = row["subject_id"]
    file_id = row["file_id"]
    label = 1 if row["label_type"] == "fall" else 0
    token_path = f"tokens/w3_k60_s1/{subject_id}__{file_id}.npy"
    tokens = np.load(token_path)
    # histogram
    hist = np.bincount(tokens, minlength=K)
    hist = hist / np.sum(hist)  # normalize
    X_hist.append(hist)
    y_rec.append(label)
X_hist = np.vstack(X_hist)
y_rec = np.array(y_rec)

print(X_hist.shape, y_rec.shape)

(60, 60) (60,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_hist, y_rec, test_size=0.25, random_state=42
)

In [6]:
rf = RandomForestClassifier(n_estimators=200)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)

print("Recording-level token model")
print("Accuracy:", accuracy_score(y_test, preds))
print("F1:", f1_score(y_test, preds))

Recording-level token model
Accuracy: 0.9333333333333333
F1: 0.9473684210526315


In [7]:
X_feat_rec = []
y_rec = []

for _, row in index_df.iterrows():
    feats = np.load(row["feature_path"])
    label = 1 if row["label_type"] == "fall" else 0
    # summarise across windows
    feat_mean = np.mean(feats, axis=0)
    feat_std = np.std(feats, axis=0)
    feat_vec = np.concatenate([feat_mean, feat_std])
    X_feat_rec.append(feat_vec)
    y_rec.append(label)
X_feat_rec = np.vstack(X_feat_rec)
y_rec = np.array(y_rec)

print(X_feat_rec.shape)


(60, 44)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X_feat_rec, y_rec, test_size=0.25, random_state=42
)

rf = RandomForestClassifier(n_estimators=200)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)

print("Recording-level RAW features")
print("Accuracy:", accuracy_score(y_test, preds))
print("F1:", f1_score(y_test, preds))


Recording-level RAW features
Accuracy: 0.8
F1: 0.8235294117647058
